# LieselVI Showcase

`LieselVI` is the high-level builder for variational inference. It mirrors `LieselOptim`, but constructs an ELBO and optimizes variational parameters.

In [1]:
import jax
import jax.numpy as jnp
import optax
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.experimental.optim as opt
import liesel.model as lsl


def normal_model(n=28, *, seed=1, name="y"):
    key = jax.random.key(seed)
    y_values = 0.9 + 0.35 * jax.random.normal(key, (n,))
    loc = lsl.Var.new_param(jnp.array(0.0), name="loc")
    y = lsl.Var.new_obs(
        y_values,
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name=name,
    )
    return lsl.Model([y])


def two_branch_model(*, seed=10):
    key1, key2 = jax.random.split(jax.random.key(seed))
    loc = lsl.Var.new_param(jnp.array(0.0), name="loc")
    y1 = lsl.Var.new_obs(
        1.0 + 0.25 * jax.random.normal(key1, (26,)),
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name="y_long",
    )
    y2 = lsl.Var.new_obs(
        -0.4 + 0.40 * jax.random.normal(key2, (10,)),
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name="y_short",
    )
    return lsl.Model([y1, y2])

## Default Builder Configuration

The default variational family is diagonal multivariate normal. Internally constructed ELBOs are scaled by default.

In [2]:
model = normal_model(seed=1)
vi = opt.LieselVI(
    model,
    stopper=opt.Stopper(epochs=5, patience=5),
    nsamples=2,
    nsamples_validate=4,
    seed=1,
)
engine = vi.build_engine()

{
    "engine_type": type(engine).__name__,
    "elbo_type": type(engine.loss).__name__,
    "loss_scaled": engine.loss.scale,
    "q_parameters": engine.loss.q.parameters,
    "optimizer_ids": [optimizer.identifier for optimizer in engine.optimizers],
}

{'engine_type': 'OptimEngine',
 'elbo_type': 'Elbo',
 'loss_scaled': True,
 'q_parameters': mappingproxy({'(loc)_loc': Var(name="(loc)_loc"),
               'h((loc)_scale)': Var(name="h((loc)_scale)")}),
 'optimizer_ids': ['000']}

## Short Fit

Use small sample counts for quick exploration. Larger values are typically useful for final runs.

In [3]:
fit_engine = opt.LieselVI(
    normal_model(seed=2),
    stopper=opt.Stopper(epochs=4, patience=4),
    nsamples=1,
    nsamples_validate=2,
    seed=2,
).build_engine()
result = fit_engine.fit()

{
    "result_type": type(result).__name__,
    "final_epoch": result.final_epoch,
    "best_epoch": result.best_epoch,
}

Training loss: 1.470, Validation loss: 1.470: 100%|██████████| 4/4 [00:00<00:00,  7.29it/s]


{'result_type': 'OptimResult', 'final_epoch': 4, 'best_epoch': 2}

## Standard ELBO Families

The string shortcuts build common multivariate normal variational families.

In [4]:
families = {}
for family in ["mvn_diag", "mvn_tril", "mvn_blocked"]:
    family_vi = opt.LieselVI(
        normal_model(seed=3),
        elbo=family,
        stopper=opt.Stopper(epochs=3, patience=3),
        nsamples=2,
        seed=3,
    )
    family_engine = family_vi.build_engine()
    families[family] = {
        "vdist_type": type(family_engine.loss.vdist).__name__,
        "q_parameter_count": len(family_engine.loss.q.parameters),
    }

families

{'mvn_diag': {'vdist_type': 'VDist', 'q_parameter_count': 2},
 'mvn_tril': {'vdist_type': 'VDist', 'q_parameter_count': 2},
 'mvn_blocked': {'vdist_type': 'CompositeVDist', 'q_parameter_count': 2}}

## Custom ELBO

Custom ELBO objects are passed through unchanged. This is the right place for Laplace-style or custom variational initialization.

In [5]:
custom_model = normal_model(seed=4)
custom_elbo = opt.Elbo.mvn_diag(
    custom_model,
    nsamples=2,
    nsamples_validate=4,
    scale_diag="laplace",
    scale_diag_bijector=None,
)
custom_vi = opt.LieselVI(
    custom_model,
    elbo=custom_elbo,
    stopper=opt.Stopper(epochs=4, patience=4),
    seed=4,
)
custom_engine = custom_vi.build_engine()

{
    "same_elbo_object": custom_engine.loss is custom_elbo,
    "q_parameters": custom_engine.loss.q.parameters,
    "loss_scaled": custom_engine.loss.scale,
}

{'same_elbo_object': True,
 'q_parameters': mappingproxy({'(loc)_scale': Var(name="(loc)_scale"),
               '(loc)_loc': Var(name="(loc)_loc")}),
 'loss_scaled': False}

## Mini-Batch VI

Passing `batch_size` constructs mini-batches from the training split.

In [6]:
mini_engine = opt.LieselVI(
    normal_model(seed=5),
    batch_size=8,
    stopper=opt.Stopper(epochs=4, patience=4),
    nsamples=2,
    seed=5,
).build_engine()

{
    "batch_type": type(mini_engine.batches).__name__,
    "batch_size": mini_engine.batches.batch_size,
    "n_full_batches": mini_engine.batches.n_full_batches,
}

{'batch_type': 'Batches', 'batch_size': 8, 'n_full_batches': 3}

## Multi-Size Observed Branches

Like `LieselOptim`, the VI builder uses manager objects automatically when observed branches have different sample sizes.

In [7]:
multi_engine = opt.LieselVI(
    two_branch_model(seed=6),
    batch_size=6,
    stopper=opt.Stopper(epochs=4, patience=4),
    nsamples=2,
    seed=6,
).build_engine()

{
    "split_type": type(multi_engine.split).__name__,
    "batch_type": type(multi_engine.batches).__name__,
    "n_trains": multi_engine.split.n_trains,
    "n_full_batches": multi_engine.batches.n_full_batches,
}

{'split_type': 'PositionSplitManager',
 'batch_type': 'BatchManager',
 'n_trains': (10, 26),
 'n_full_batches': 4}

## Explicit Optimizer Sequences

Adam is the only built-in VI optimizer shortcut, but explicit optimizer objects are supported.

In [8]:
explicit_model = normal_model(seed=7)
explicit_elbo = opt.Elbo.mvn_diag(explicit_model, nsamples=2, nsamples_validate=4)
explicit_optimizer = opt.Optimizer(
    list(explicit_elbo.q.parameters),
    optax.sgd(0.01),
    identifier="q_sgd",
)
explicit_engine = opt.LieselVI(
    explicit_model,
    elbo=explicit_elbo,
    optimizers=[explicit_optimizer],
    stopper=opt.Stopper(epochs=3, patience=3),
    seed=7,
).build_engine()

{
    "optimizer_ids": [optimizer.identifier for optimizer in explicit_engine.optimizers],
    "optimized_keys": explicit_engine.position_keys,
}

{'optimizer_ids': ['q_sgd'], 'optimized_keys': ['(loc)_loc', 'h((loc)_scale)']}